In [1]:
%%writefile graph.h
// graph.h -- two fixed CSR graph inputs, and the CPU reference BFS.
// Do NOT change this file

#pragma once
#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <queue>
#include <utility>
#include <vector>

struct CSRGraph {
    int V = 0;                  // number of vertices
    long long E = 0;            // number of directed edges (each undirected edge stored twice)
    std::vector<int> rowPtr;    // size V+1: neighbors of v are colIdx[rowPtr[v] .. rowPtr[v+1]-1]
    std::vector<int> colIdx;    // size E
};

// Build an undirected CSR graph from an edge list
static CSRGraph buildCSR(int V, std::vector<std::pair<int, int>>& edges) {
    std::vector<std::pair<int, int>> both;
    both.reserve(edges.size() * 2);
    for (auto& e : edges)
        if (e.first != e.second) { both.push_back(e); both.push_back({e.second, e.first}); }
    std::sort(both.begin(), both.end());
    both.erase(std::unique(both.begin(), both.end()), both.end());

    CSRGraph g;
    g.V = V;
    g.E = (long long)both.size();
    g.rowPtr.assign(V + 1, 0);
    g.colIdx.resize(both.size());
    for (auto& e : both) g.rowPtr[e.first + 1]++;
    for (int v = 0; v < V; ++v) g.rowPtr[v + 1] += g.rowPtr[v];
    for (size_t i = 0; i < both.size(); ++i) g.colIdx[i] = both[i].second;
    return g;
}

// Input 1:  a road network or a mesh.
static CSRGraph makeGrid(int W, int H) {
    std::vector<std::pair<int, int>> edges;
    edges.reserve(2LL * W * H);
    for (int y = 0; y < H; ++y)
        for (int x = 0; x < W; ++x) {
            int v = y * W + x;
            if (x + 1 < W) edges.push_back({v, v + 1});
            if (y + 1 < H) edges.push_back({v, v + W});
        }
    return buildCSR(W * H, edges);
}

// Input 2: an R-MAT (social networks) graph with Graph500 parameters, skewed degrees, small diameter
static CSRGraph makeRMAT(int scale, int edgeFactor, uint64_t seed) {
    const int V = 1 << scale;
    const double a = 0.57, b = 0.19, c = 0.19;      // d = 0.05
    uint64_t s = seed;
    auto rnd = [&s]() {                              // xorshift64*, uniform in [0,1)
        s ^= s >> 12; s ^= s << 25; s ^= s >> 27;
        return (double)((s * 2685821657736338717ULL) >> 11) * (1.0 / 9007199254740992.0);
    };
    std::vector<std::pair<int, int>> edges;
    edges.reserve((size_t)V * edgeFactor);
    for (long long i = 0; i < (long long)V * edgeFactor; ++i) {
        int u = 0, v = 0;
        for (int bit = scale - 1; bit >= 0; --bit) {
            double r = rnd();
            if (r < a) {}
            else if (r < a + b) v |= 1 << bit;
            else if (r < a + b + c) u |= 1 << bit;
            else { u |= 1 << bit; v |= 1 << bit; }
        }
        edges.push_back({u, v});
    }
    return buildCSR(V, edges);
}

// Your GPU kernel should complete with this CPU kernel
// Use it to check correctness
static int bfsCPU(const CSRGraph& g, int src, std::vector<int>& level) {
    level.assign(g.V, -1);
    std::queue<int> q;
    level[src] = 0;
    q.push(src);
    int maxLevel = 0;
    while (!q.empty()) {
        int v = q.front(); q.pop();
        for (int i = g.rowPtr[v]; i < g.rowPtr[v + 1]; ++i) {
            int u = g.colIdx[i];
            if (level[u] == -1) {
                level[u] = level[v] + 1;
                maxLevel = std::max(maxLevel, level[u]);
                q.push(u);
            }
        }
    }
    return maxLevel + 1;
}

static bool verify(const std::vector<int>& ref, const std::vector<int>& got) {
    for (size_t v = 0; v < ref.size(); ++v)
        if (ref[v] != got[v]) {
            printf("MISMATCH at vertex %zu: CPU level %d, GPU level %d\n", v, ref[v], got[v]);
            return false;
        }
    return true;
}

static CSRGraph loadInput(const char* name) {
    if (strcmp(name, "grid") == 0) return makeGrid(1024, 1024);      // 1,048,576 vertices
    if (strcmp(name, "rmat") == 0) return makeRMAT(18, 16, 2360);    // 262,144 vertices
    printf("unknown input '%s' (use grid or rmat)\n", name);
    return CSRGraph{};
}

static double nowMs() {
    using namespace std::chrono;
    return duration<double, std::milli>(steady_clock::now().time_since_epoch()).count();
}

Writing graph.h


In [2]:
%%writefile bfs.cu
// You are free to modify this file
// but do not change the function signatures of bfsLevelKernel or bfsGPU.

#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <vector>
#include "graph.h"

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err_ = (call);                                              \
        if (err_ != cudaSuccess) {                                              \
            printf("CUDA error %s at %s:%d\n", cudaGetErrorString(err_),        \
                   __FILE__, __LINE__);                                         \
            exit(1);                                                            \
        }                                                                       \
    } while (0)

// ---------------------------------------------------------------------------
// TODO (1): one level of a level-synchronous BFS.
//
//   One thread per vertex v. If v is on the current frontier, visit every neighbor u of v:
//       neighbors of v are colIdx[rowPtr[v]] ... colIdx[rowPtr[v+1] - 1]
//   If u is unvisited (level[u] == -1), set level[u] = curLevel + 1 so that u will be on the next frontier
//  If any vertex was newly visited, set *changed = 1 so that the host (in TODO(2)) knows to continue to the next level.
// ---------------------------------------------------------------------------
__global__ void bfsLevelKernel(const int* rowPtr, const int* colIdx, int* level,
                               int V, int curLevel, int* changed) {
    // Map each thread to a unique vertex ID v
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= V) return;

    // Only vertices on the current BFS frontier expand their neighbors
    if (level[v] == curLevel) {
        int startEdge = rowPtr[v];
        int endEdge = rowPtr[v + 1];
        for (int e = startEdge; e < endEdge; ++e) {
            int u = colIdx[e];
            // Idempotent write: all threads discovering u in this level write the exact same value (curLevel + 1)
            if (level[u] == -1) {
                level[u] = curLevel + 1;
                *changed = 1;
            }
        }
    }
}

float bfsGPU(const CSRGraph& g, int src, std::vector<int>& level, int& numLevels) {
    int *dRowPtr, *dColIdx, *dLevel, *dChanged;
    CUDA_CHECK(cudaMalloc(&dRowPtr, (g.V + 1) * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&dColIdx, g.E * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&dLevel, g.V * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&dChanged, sizeof(int)));
    CUDA_CHECK(cudaMemcpy(dRowPtr, g.rowPtr.data(), (g.V + 1) * sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dColIdx, g.colIdx.data(), g.E * sizeof(int), cudaMemcpyHostToDevice));

    // All vertices unvisited (-1), except the source (level 0).
    level.assign(g.V, -1);
    level[src] = 0;
    CUDA_CHECK(cudaMemcpy(dLevel, level.data(), g.V * sizeof(int), cudaMemcpyHostToDevice));

    // -----------------------------------------------------------------------
    // TODO (2): the level loop on the host.
    //   curLevel = 0
    //   repeat:
    //     set *dChanged = 0                      (cudaMemset or cudaMemcpy)
    //     launch bfsLevelKernel with enough blocks of 256 threads to cover V
    //     copy dChanged back to the host
    //     curLevel++
    //   until nothing changed
    //   numLevels = number of levels that contain at least one vertex
    // -----------------------------------------------------------------------
    const int threadsPerBlock = 256;
    const int blocksPerGrid = (g.V + threadsPerBlock - 1) / threadsPerBlock;

    cudaEvent_t startEvent, stopEvent;
    CUDA_CHECK(cudaEventCreate(&startEvent));
    CUDA_CHECK(cudaEventCreate(&stopEvent));

    int curLevel = 0;
    int hChanged = 0;

    // Record start time of the GPU level-synchronous kernel loop
    CUDA_CHECK(cudaEventRecord(startEvent));
    do {
        hChanged = 0;
        CUDA_CHECK(cudaMemset(dChanged, 0, sizeof(int)));

        bfsLevelKernel<<<blocksPerGrid, threadsPerBlock>>>(
            dRowPtr, dColIdx, dLevel, g.V, curLevel, dChanged);
        CUDA_CHECK(cudaGetLastError());

        CUDA_CHECK(cudaMemcpy(&hChanged, dChanged, sizeof(int), cudaMemcpyDeviceToHost));
        if (hChanged) {
            curLevel++;
        }
    } while (hChanged != 0);

    // Record end time after all frontier levels have completed
    CUDA_CHECK(cudaEventRecord(stopEvent));
    CUDA_CHECK(cudaEventSynchronize(stopEvent));

    // Levels 0 .. curLevel each contain at least one vertex
    numLevels = curLevel + 1;

    // -----------------------------------------------------------------------
    // TODO (3):
    // copy dLevel back into level.
    // print the timing information and return the elapsed time
    // -----------------------------------------------------------------------
    float ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, startEvent, stopEvent));
    CUDA_CHECK(cudaMemcpy(level.data(), dLevel, g.V * sizeof(int), cudaMemcpyDeviceToHost));

    printf("GPU BFS kernel loop elapsed: %.3f ms (%d kernel launches)\n", ms, numLevels);

    CUDA_CHECK(cudaEventDestroy(startEvent));
    CUDA_CHECK(cudaEventDestroy(stopEvent));
    cudaFree(dRowPtr);
    cudaFree(dColIdx);
    cudaFree(dLevel);
    cudaFree(dChanged);
    return ms;
}

int main(int argc, char** argv) {
    const char* input = argc > 1 ? argv[1] : "grid";
    // Optional second argument: number of runs to compute median time (default = 1)
    const int runs = argc > 2 ? std::max(1, std::atoi(argv[2])) : 1;

    CSRGraph g = loadInput(input);
    if (g.V == 0) return 1;
    const int src = 0;
    printf("Input %s: %d vertices, %lld directed edges (avg degree %.1f, max degree %d)\n",
           input, g.V, g.E, (double)g.E / g.V,
           [&] { int m = 0; for (int v = 0; v < g.V; ++v) m = std::max(m, g.rowPtr[v + 1] - g.rowPtr[v]); return m; }());

    std::vector<int> refLevel, gpuLevel;
    std::vector<double> cpuTimes, gpuTimes;
    int cpuLevels = 0, gpuLevels = 0;
    bool ok = true;

    for (int r = 0; r < runs; ++r) {
        double c0 = nowMs();
        cpuLevels = bfsCPU(g, src, refLevel);
        double cpuMs = nowMs() - c0;

        float gpuMs = bfsGPU(g, src, gpuLevel, gpuLevels);

        cpuTimes.push_back(cpuMs);
        gpuTimes.push_back(static_cast<double>(gpuMs));
        ok = ok && verify(refLevel, gpuLevel);
    }

    // Compute median times across runs
    std::sort(cpuTimes.begin(), cpuTimes.end());
    std::sort(gpuTimes.begin(), gpuTimes.end());
    double medCpuMs = (runs % 2 == 1)
                          ? cpuTimes[runs / 2]
                          : 0.5 * (cpuTimes[runs / 2 - 1] + cpuTimes[runs / 2]);
    double medGpuMs = (runs % 2 == 1)
                          ? gpuTimes[runs / 2]
                          : 0.5 * (gpuTimes[runs / 2 - 1] + gpuTimes[runs / 2]);

    long long reached = 0;
    for (int l : refLevel) reached += (l >= 0);
    printf("CPU BFS: %8.2f ms, %d levels, %lld vertices reached\n", medCpuMs, cpuLevels, reached);
    printf("GPU BFS: %8.2f ms, %d levels\n", medGpuMs, gpuLevels);

    printf("Result: %s", ok ? "PASS" : "FAIL");
    if (ok) printf("   speedup %.2fx, GPU %.1f MTEPS", medCpuMs / medGpuMs, g.E / (medGpuMs * 1e3));
    printf("\n");
    if (runs > 1) {
        printf("(Median over %d runs: CPU = %.3f ms, GPU = %.3f ms)\n", runs, medCpuMs, medGpuMs);
    }
    return ok ? 0 : 1;
}

Writing bfs.cu


In [3]:
%%bash
nvcc -O3 -lineinfo -arch=sm_75 bfs.cu -o bfs
./bfs grid
./bfs rmat

Input grid: 1048576 vertices, 4190208 directed edges (avg degree 4.0, max degree 4)
GPU BFS kernel loop elapsed: 84.828 ms (2047 kernel launches)
CPU BFS:    59.72 ms, 2047 levels, 1048576 vertices reached
GPU BFS:    84.83 ms, 2047 levels
Result: PASS   speedup 0.70x, GPU 49.4 MTEPS
Input rmat: 262144 vertices, 7611674 directed edges (avg degree 29.0, max degree 25165)
GPU BFS kernel loop elapsed: 12.103 ms (5 kernel launches)
CPU BFS:    28.34 ms, 5 levels, 174050 vertices reached
GPU BFS:    12.10 ms, 5 levels
Result: PASS   speedup 2.34x, GPU 628.9 MTEPS


In [4]:
%%bash
echo "=== 10 Runs Median: grid ==="
./bfs grid 10
echo "=== 10 Runs Median: rmat ==="
./bfs rmat 10

=== 10 Runs Median: grid ===
Input grid: 1048576 vertices, 4190208 directed edges (avg degree 4.0, max degree 4)
GPU BFS kernel loop elapsed: 89.159 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 91.181 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 87.747 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 85.757 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 78.829 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 79.058 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 79.156 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 73.449 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 74.795 ms (2047 kernel launches)
GPU BFS kernel loop elapsed: 72.490 ms (2047 kernel launches)
CPU BFS:    62.60 ms, 2047 levels, 1048576 vertices reached
GPU BFS:    79.11 ms, 2047 levels
Result: PASS   speedup 0.79x, GPU 53.0 MTEPS
(Median over 10 runs: CPU = 62.600 ms, GPU = 79.107 ms)
=== 10 Runs Median: rmat ===
Input rmat: 262144 vertices, 7611674 direct

In [5]:
%%bash
METRICS="gpu__time_duration.sum,dram__bytes_read.sum,dram__throughput.avg.pct_of_peak_sustained_elapsed,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum,lts__t_sector_hit_rate.pct"

echo "=== NCU Profile: grid (launch #1024, peak frontier) ==="
ncu --kernel-name bfsLevelKernel --launch-skip 1023 --launch-count 1 \
    --metrics $METRICS \
    ./bfs grid

echo "=== NCU Profile: rmat (launch #4, peak frontier) ==="
ncu --kernel-name bfsLevelKernel --launch-skip 3 --launch-count 1 \
    --metrics $METRICS \
    ./bfs rmat

=== NCU Profile: grid (launch #1024, peak frontier) ===
==PROF== Connected to process 2084 (/content/bfs)
==PROF== Profiling "bfsLevelKernel": 0%....50%....100% - 3 passes
Input grid: 1048576 vertices, 4190208 directed edges (avg degree 4.0, max degree 4)
GPU BFS kernel loop elapsed: 1883.938 ms (2047 kernel launches)
CPU BFS:    57.20 ms, 2047 levels, 1048576 vertices reached
GPU BFS:  1883.94 ms, 2047 levels
Result: PASS   speedup 0.03x, GPU 2.2 MTEPS
==PROF== Disconnected from process 2084
[2084] bfs@127.0.0.1
  bfsLevelKernel(const int *, const int *, int *, int, int, int *) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    -------------------------------------------------- ----------- ------------
    Metric Name                                        Metric Unit Metric Value
    -------------------------------------------------- ----------- ------------
    dram__bytes_read.sum                                     Mbyte 